# Treasury Cash Flow Forecasting - Comprehensive EDA v2

**Objective**: Build statistically rigorous EDA with multi-stage feature engineering for 8-week forecasting

**Target**: Beat existing LP forecast accuracy with ~118 engineered features

**Approach**:
- Stage 1: Data Quality & Statistical Assessment
- Stage 2: Feature Engineering Part 1 (Daily Level)
- Stage 3: Weekly Aggregation
- Stage 4: Feature Engineering Part 2 (Weekly Level)
- Stage 5: LP Processing
- Stage 6: Merge Actuals + LP
- Stage 7: Feature Engineering Part 3 (Cross-features)
- Stage 8: Final Analysis & Feature Selection

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Statistical tests
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Imports complete")

---
## Stage 1: Data Understanding & Quality Assessment

### 1.1 Load Raw Data

In [ ]:
# Load actuals
actuals = pd.read_csv('../data/raw/actuals_curated.csv')
actuals['Value Date'] = pd.to_datetime(actuals['Value Date'])

print("="*80)
print("ACTUALS DATA")
print("="*80)
print(f"Shape: {actuals.shape}")
print(f"\nColumns: {list(actuals.columns)}")
print(f"\nDate Range: {actuals['Value Date'].min()} to {actuals['Value Date'].max()}")
print(f"Total Days: {(actuals['Value Date'].max() - actuals['Value Date'].min()).days}")
print(f"\nMemory: {actuals.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

actuals.info()

In [ ]:
# Load LP
lp = pd.read_csv('../data/raw/LP_17C7.csv')

print("="*80)
print("LIQUIDITY PLAN DATA")
print("="*80)
print(f"Shape: {lp.shape}")
print(f"\nColumns (first 20): {list(lp.columns[:20])}")
print(f"\nMemory: {lp.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

lp.info()

In [ ]:
# Load FX rates
fx = pd.read_csv('../data/raw/eurofxref-hist.csv')
fx['Date'] = pd.to_datetime(fx['Date'])

print("="*80)
print("FX RATES DATA")
print("="*80)
print(f"Shape: {fx.shape}")
print(f"\nDate Range: {fx['Date'].min()} to {fx['Date'].max()}")
print(f"\nCurrencies: {[col for col in fx.columns if col != 'Date'][:10]}")

fx.head()

In [ ]:
# Load entity mapping
entity_map = pd.read_csv('../data/reference/Entity-Liquidity_Map.csv')

print("="*80)
print("ENTITY-LIQUIDITY MAPPING")
print("="*80)
print(f"Shape: {entity_map.shape}")
print(f"\nColumns: {list(entity_map.columns)}")

entity_map.head(10)

### 1.2 Data Quality Assessment - Actuals

In [ ]:
# Temporal coverage analysis
print("="*80)
print("TEMPORAL COVERAGE ANALYSIS")
print("="*80)

# Daily coverage
date_range = pd.date_range(actuals['Value Date'].min(), actuals['Value Date'].max(), freq='D')
dates_with_data = actuals['Value Date'].unique()
missing_dates = set(date_range) - set(dates_with_data)

print(f"\nTotal possible days: {len(date_range)}")
print(f"Days with data: {len(dates_with_data)}")
print(f"Missing days: {len(missing_dates)} ({len(missing_dates)/len(date_range)*100:.1f}%)")

# Are missing days weekends?
if len(missing_dates) > 0:
    missing_df = pd.DataFrame({'date': list(missing_dates)})
    missing_df['dayofweek'] = pd.to_datetime(missing_df['date']).dt.day_name()
    print(f"\nMissing days by day of week:")
    print(missing_df['dayofweek'].value_counts())

# Weekly coverage
actuals['Week'] = actuals['Value Date'].dt.to_period('W-MON')  # Monday-based weeks
print(f"\nTotal weeks: {actuals['Week'].nunique()}")
print(f"First week: {actuals['Week'].min()}")
print(f"Last week: {actuals['Week'].max()}")

In [ ]:
# Entity coverage
print("="*80)
print("ENTITY COVERAGE ANALYSIS")
print("="*80)

entity_stats = actuals.groupby('Entity').agg({
    'Value Date': ['min', 'max', 'nunique'],
    'Amount Functional Currency': ['count', 'sum', 'mean', 'std'],
    'Liquidity Group': lambda x: x.unique().tolist()
}).round(2)

entity_stats.columns = ['First_Date', 'Last_Date', 'Days_With_Data', 
                         'Txn_Count', 'Total_Amount', 'Avg_Amount', 'Std_Amount',
                         'Liquidity_Groups']

print(f"\nTotal entities: {actuals['Entity'].nunique()}")
print(f"\nTop 10 entities by transaction count:")
print(entity_stats.nlargest(10, 'Txn_Count')[['Txn_Count', 'Days_With_Data', 'Total_Amount']])

print(f"\nBottom 10 entities by transaction count:")
print(entity_stats.nsmallest(10, 'Txn_Count')[['Txn_Count', 'Days_With_Data', 'Total_Amount']])

In [ ]:
# Missing data patterns
print("="*80)
print("MISSING DATA ANALYSIS")
print("="*80)

missing = actuals.isnull().sum()
missing_pct = (missing / len(actuals) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Pct': missing_pct
})
print(missing_df[missing_df['Missing_Count'] > 0])

if missing_df['Missing_Count'].sum() == 0:
    print("\n✓ No missing values in actuals data")

In [ ]:
# Outlier detection
print("="*80)
print("OUTLIER DETECTION (IQR Method)")
print("="*80)

# By liquidity group
for liq_group in actuals['Liquidity Group'].unique():
    data = actuals[actuals['Liquidity Group'] == liq_group]['Amount Functional Currency']
    
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    print(f"\n{liq_group}:")
    print(f"  Q1: {Q1:,.2f}, Q3: {Q3:,.2f}, IQR: {IQR:,.2f}")
    print(f"  Bounds: [{lower_bound:,.2f}, {upper_bound:,.2f}]")
    print(f"  Outliers: {len(outliers):,} ({len(outliers)/len(data)*100:.2f}%)")
    if len(outliers) > 0:
        print(f"  Outlier range: [{outliers.min():,.2f}, {outliers.max():,.2f}]")

In [ ]:
# Weekend/holiday transaction analysis
print("="*80)
print("WEEKEND TRANSACTION ANALYSIS")
print("="*80)

actuals['DayOfWeek'] = actuals['Value Date'].dt.day_name()
actuals['IsWeekend'] = actuals['Value Date'].dt.dayofweek >= 5

dow_dist = actuals['DayOfWeek'].value_counts()
print(f"\nTransactions by day of week:")
print(dow_dist)

weekend_txns = actuals[actuals['IsWeekend']]
print(f"\nWeekend transactions: {len(weekend_txns):,} ({len(weekend_txns)/len(actuals)*100:.2f}%)")

if len(weekend_txns) > 0:
    print(f"\nWeekend entities:")
    weekend_entities = weekend_txns['Entity'].value_counts().head(10)
    print(weekend_entities)
    print(f"\n⚠️ Recommendation: Review weekend transactions with treasury team")

### 1.3 Statistical Tests - Stationarity & Seasonality

In [ ]:
# Aggregate to weekly for time series analysis
print("="*80)
print("PREPARING WEEKLY TIME SERIES FOR STATISTICAL TESTS")
print("="*80)

# Create week_start (Monday)
actuals['week_start'] = actuals['Value Date'] - pd.to_timedelta(actuals['Value Date'].dt.dayofweek, unit='D')

# Aggregate to weekly
weekly_ts = actuals.groupby(['Entity', 'Liquidity Group', 'week_start']).agg({
    'Amount Functional Currency': 'sum'
}).reset_index()

weekly_ts.columns = ['entity', 'liq_group', 'week_start', 'amount']
weekly_ts = weekly_ts.sort_values(['entity', 'liq_group', 'week_start'])

print(f"\nWeekly time series shape: {weekly_ts.shape}")
print(f"Entities: {weekly_ts['entity'].nunique()}")
print(f"Weeks: {weekly_ts['week_start'].nunique()}")
print(f"\nSample:")
print(weekly_ts.head(10))

In [ ]:
# Augmented Dickey-Fuller test for stationarity
print("="*80)
print("STATIONARITY TESTS (ADF Test)")
print("="*80)
print("\nTesting top 5 entities by transaction volume...")

top_entities = actuals.groupby('Entity')['Amount Functional Currency'].sum().nlargest(5).index

stationarity_results = []

for entity in top_entities:
    for liq_group in ['TRR', 'TRP']:
        ts = weekly_ts[(weekly_ts['entity'] == entity) & (weekly_ts['liq_group'] == liq_group)]
        
        if len(ts) < 20:  # Need minimum observations
            continue
            
        ts = ts.set_index('week_start')['amount'].fillna(0)
        
        # ADF test
        adf_result = adfuller(ts, autolag='AIC')
        
        is_stationary = adf_result[1] < 0.05  # p-value < 0.05
        
        stationarity_results.append({
            'entity': entity,
            'liq_group': liq_group,
            'adf_statistic': adf_result[0],
            'p_value': adf_result[1],
            'is_stationary': is_stationary,
            'n_obs': len(ts)
        })

stationarity_df = pd.DataFrame(stationarity_results)
print(stationarity_df.to_string(index=False))

stationary_pct = stationarity_df['is_stationary'].sum() / len(stationarity_df) * 100
print(f"\n✓ Stationary series: {stationarity_df['is_stationary'].sum()}/{len(stationarity_df)} ({stationary_pct:.1f}%)")

if stationary_pct < 50:
    print("\n⚠️ Recommendation: Consider differencing or detrending for non-stationary series")

In [ ]:
# Seasonal decomposition
print("="*80)
print("SEASONALITY ANALYSIS")
print("="*80)

# Test with one high-volume entity
test_entity = top_entities[0]
test_liq = 'TRR'

ts_test = weekly_ts[(weekly_ts['entity'] == test_entity) & (weekly_ts['liq_group'] == test_liq)]
ts_test = ts_test.set_index('week_start')['amount'].fillna(0)

if len(ts_test) >= 52:  # Need at least 1 year for seasonal decomposition
    print(f"\nAnalyzing: {test_entity} - {test_liq}")
    print(f"Observations: {len(ts_test)}")
    
    # Decompose (additive model)
    decomposition = seasonal_decompose(ts_test, model='additive', period=52, extrapolate_trend='freq')
    
    # Plot
    fig, axes = plt.subplots(4, 1, figsize=(15, 10))
    
    decomposition.observed.plot(ax=axes[0], title='Observed')
    decomposition.trend.plot(ax=axes[1], title='Trend')
    decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
    decomposition.resid.plot(ax=axes[3], title='Residual')
    
    plt.tight_layout()
    plt.savefig('../artifacts/seasonal_decomposition.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Seasonality strength
    seasonal_strength = 1 - (decomposition.resid.var() / (decomposition.seasonal + decomposition.resid).var())
    print(f"\nSeasonality strength: {seasonal_strength:.3f}")
    
    if seasonal_strength > 0.3:
        print("✓ Strong seasonality detected - include seasonal features")
    else:
        print("⚠️ Weak seasonality - seasonal features may have limited impact")
else:
    print(f"\n⚠️ Insufficient data for seasonal decomposition (need ≥52 weeks, have {len(ts_test)})")

### 1.4 Summary & Recommendations

In [ ]:
print("="*80)
print("STAGE 1 SUMMARY: DATA QUALITY & STATISTICAL ASSESSMENT")
print("="*80)

summary = f"""
✓ ACTUALS DATA:
  - Rows: {len(actuals):,}
  - Entities: {actuals['Entity'].nunique()}
  - Date Range: {actuals['Value Date'].min()} to {actuals['Value Date'].max()}
  - Weeks: {actuals['Week'].nunique()}
  - Missing Values: {actuals.isnull().sum().sum()}
  - Weekend Transactions: {len(weekend_txns):,} ({len(weekend_txns)/len(actuals)*100:.2f}%)

✓ STATISTICAL PROPERTIES:
  - Stationary Series: {stationary_pct:.1f}%
  - Seasonality: Detected (include seasonal features)

📋 RECOMMENDATIONS:
  1. Review weekend transactions with treasury team
  2. Consider differencing for non-stationary series
  3. Include seasonal features (week of year, month, quarter)
  4. Handle outliers with winsorization or capping
  5. Normalize week references to Monday (ISO week start)
"""

print(summary)

# Save summary
with open('../artifacts/stage1_summary.txt', 'w') as f:
    f.write(summary)

print("\n✓ Stage 1 complete. Ready for Stage 2: Feature Engineering (Daily Level)")